## Deep Agents
`create_agent` daje pętlę wywołań narzędzi. Przy długich, wieloetapowych zadaniach (research, pisanie raportu, programowanie) agent potrzebuje też miejsca na notatki, sposobu na delegowanie podzadań i ochrony przed przepełnieniem kontekstu. **Deep Agents** (`deepagents`) to gotowa implementacja agenta zbudowana na LangChain i LangGraph, która dodaje właśnie to:
- **wirtualny system plików** – narzędzia `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`; pliki są przechowywane w stanie agenta,
- **subagenci** – narzędzie `task` deleguje pracę do wyspecjalizowanych agentów z własnym promptem i narzędziami, dzięki czemu główny kontekst pozostaje mały,
- **zarządzanie kontekstem** – długie rozmowy są automatycznie streszczane (`SummarizationMiddleware`).

Wynikiem `create_deep_agent` jest zwykły skompilowany graf LangGraph, więc `invoke`, `stream`, checkpointery i LangSmith działają jak zwykle.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/pl/13_1_Deep_agents.ipynb)

| | |
|---|---|
| 🎯 **Czego się nauczysz** | Deep Agents: gotowa „uprząż” agenta z wirtualnym systemem plików, subagentami i zarządzaniem kontekstem, zbudowana na LangGraph. |
| 🔑 **Klucze API** | `OPENAI_API_KEY`, `TAVILY_API_KEY` |
| 💲 **Koszt API** | niski – kilkanaście zapytań |
| ⏱️ **Czas wykonania** | ~2 min |

### Instalacja bibliotek

In [1]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU deepagents langchain-openai langchain-tavily python-dotenv
# Dla wyszukiwania Tavily dodaj też TAVILY_API_KEY w Colab Secrets lub lokalnym pliku .env.

Note: you may need to restart the kernel to use updated packages.


### Wczytanie zmiennych środowiskowych

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 1. Najprostszy deep agent i jego system plików
Nawet bez własnych narzędzi agent ma wbudowane narzędzia plikowe. Prosimy go o zapisanie wyniku w pliku, a potem odczytujemy plik ze zwróconego stanu.

In [3]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="openai:gpt-5-mini",
    system_prompt="Jesteś pomocnym asystentem. Każdy wynik pracy zapisuj w pliku.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Napisz haiku o LangGraph i zapisz je w pliku /haiku.md."}]}
)

# narzędzia, które agent postanowił wywołać
for message in result["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        print("wywołanie narzędzia:", call["name"], call["args"].get("file_path", ""))

print("\nPliki w stanie agenta:", list(result["files"]))
print(result["files"]["/haiku.md"]["content"])

wywołanie narzędzia: write_file /haiku.md

Pliki w stanie agenta: ['/haiku.md']
LangGraph w sieci
łącza myśli i kodu
mapa dróg myśli



### 2. Subagenci: researcher i krytyk
Subagenta opisuje słownik: `name`, `description` (na tej podstawie główny agent decyduje, kiedy delegować), `system_prompt` i opcjonalnie `tools`. Główny agent wywołuje subagenta wbudowanym narzędziem `task`. Subagent pracuje we własnym kontekście i zwraca tylko końcową odpowiedź.

In [4]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=3)

researcher = {
    "name": "researcher",
    "description": "Wyszukuje informacje w internecie i zwraca zwięzłe fakty ze źródłami dla jednego konkretnego pytania.",
    "system_prompt": (
        "Jesteś researcherem. Użyj wyszukiwarki, aby odpowiedzieć na otrzymane pytanie. "
        "Zwróć 5-8 krótkich punktów z faktami i adresem URL źródła dla każdego z nich. Pisz po polsku."
    ),
    "tools": [web_search],
}

critic = {
    "name": "critic",
    "description": "Recenzuje szkic raportu zapisany w pliku i wypisuje konkretne poprawki.",
    "system_prompt": (
        "Jesteś wymagającym redaktorem. Przeczytaj wskazany plik i zwróć "
        "maksymalnie 5 konkretnych poprawek do wprowadzenia. Nie przepisuj raportu. Pisz po polsku."
    ),
}

analyst = create_deep_agent(
    model="openai:gpt-5-mini",
    subagents=[researcher, critic],
    system_prompt=(
        "Jesteś głównym analitykiem, który pisze krótkie raporty badawcze po polsku.\n"
        "1. Pytania badawcze deleguj do subagenta `researcher`.\n"
        "2. Zapisz szkic raportu w /report.md (maks. 300 słów, z sekcją Źródła).\n"
        "3. Poproś subagenta `critic` o recenzję /report.md.\n"
        "4. Wprowadź poprawki i zapisz wersję końcową w /report.md.\n"
        "Odpowiedz użytkownikowi dwuzdaniowym podsumowaniem."
    ),
)

### 3. Uruchom agenta i obserwuj delegowanie
Strumieniujemy dwa tryby naraz: `updates`, aby wypisać każde wywołanie narzędzia (delegowanie do subagentów i operacje na plikach), oraz `values`, aby zachować stan końcowy.

In [5]:
question = (
    "Jakie są główne różnice między create_agent z LangChain a StateGraph z LangGraph "
    "i kiedy warto użyć każdego z nich?"
)

final_state = None
for mode, chunk in analyst.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode=["updates", "values"],
):
    if mode == "values":
        final_state = chunk
        continue
    for node, update in chunk.items():
        messages = (update or {}).get("messages", [])
        messages = getattr(messages, "value", messages)  # niektóre aktualizacje opakowują listę
        for message in messages:
            for call in getattr(message, "tool_calls", None) or []:
                target = call["args"].get("subagent_type") or call["args"].get("file_path", "")
                print(f"[{node}] {call['name']} -> {target}")

print("\nOdpowiedź agenta:\n", final_state["messages"][-1].text)

[model] task -> researcher
[model] write_file -> /report.md
[model] task -> critic
[model] write_file -> /report.md

Odpowiedź agenta:
 create_agent to paradygmat agentowy: model decyduje dynamicznie, które narzędzia wywołać — dobry do interaktywnych, adaptacyjnych asystentów i szybkiego prototypowania. StateGraph z kolei wprowadza jawny, strukturalny stan i reducery, co czyni go lepszym wyborem dla deterministycznych, wieloetapowych workflowów, testowalności i kontroli nad walidacją stanu; w praktyce często warto stosować hybrydę (graf orkiestruje, agent wykonuje wybrane kroki).


### 4. Końcowy raport z wirtualnego systemu plików

In [6]:
from IPython.display import Markdown

Markdown(final_state["files"]["/report.md"]["content"])

Raport: Różnice między LangChain.create_agent a LangGraph.StateGraph

Streszczenie
create_agent (LangChain) to paradygmat agentowy: pętla model→narzędzia→model, w którym agent (model wspierany narzędziami) podejmuje dynamiczne decyzje o wywołaniach. StateGraph (LangGraph) to podejście grafowe / przypominające maszynę stanów: jawny, strukturalny stan (state_schema) i reducery prowadzą deterministyczne, wieloetapowe workflowy.

Główne różnice
- Paradygmat: agent (adaptacyjny, decyzyjny) vs state graph / state-machine (jawny stan, deterministyczne przejścia).
- Zarządzanie stanu: create_agent bazuje na pamięci i checkpointerach; StateGraph udostępnia state_schema, walidację stanu i reducery, co ułatwia typowanie i odtwarzalność.
- Integracje: oba wspierają LLM i narzędzia; agent optymalizowany pod tool-calling, StateGraph mapuje kroki jako węzły grafu.
- Skalowalność i orkiestracja: StateGraph lepszy do kompilacji i deterministycznej orkiestracji; agent szybszy do prototypu.
- Testowalność i bezpieczeństwo: StateGraph upraszcza testy jednostkowe dzięki jawnemu stanowi. Należy zastosować zabezpieczenia: walidacja wejścia (schema validation), ochrona przed prompt injection (sanity checks, prompt templates, allowlists), sandboxing narzędzi, uwierzytelnianie/ACL, rate limiting, pełne logowanie i audyt.

Rekomendacje
- Use create_agent gdy potrzebny adaptacyjny, narzędziowy agent (np. chatbot helpdesk).
- Use StateGraph dla wieloetapowych procesów biznesowych, ETL i tam, gdzie wymagana jest przewidywalność.
- Hybryda: StateGraph orkiestruje przepływ; w wybranych węzłach uruchamia się agentów (ściśle kontrolowane wejścia i sandboxing).

Źródła
- https://docs.langchain.com/oss/python/langchain/agents
- https://reference.langchain.com/python/langchain/agents/factory/create_agent
- https://reference.langchain.com/python/langgraph/graph/state/StateGraph
- https://langchain-ai.github.io/langgraphjs/reference/classes/langgraph.StateGraph.html
- https://www.langchain.com/blog/langgraph
- OWASP: https://owasp.org
- Artykuły o prompt injection i mitigations (np. https://arxiv.org/abs/2302.07273) oraz dokumentacja LangChain dotycząca bezpieczeństwa i walidacji schematów.

### Kiedy czego używać?
| Potrzeba | Narzędzie |
|---|---|
| Model + narzędzia w pętli, krótkie zadania | `langchain.agents.create_agent` |
| Długie, wieloetapowe zadania z notatkami, plikami i delegowaniem | `deepagents.create_deep_agent` |
| W pełni własny przepływ sterowania (stałe kroki, pętle, rozgałęzienia) | LangGraph `StateGraph` |

Dalsze kroki: dodaj agentowi pamięć długoterminową (`memory=`), umiejętności (`skills=`), akceptację człowieka dla wybranych narzędzi (`interrupt_on=`) albo prawdziwy system plików / sandbox przez inny `backend`.